# 🛡️ Rakshak AI - Training (Public Datasets - No Kaggle Required)

**Updated Version**: Uses publicly accessible datasets with direct download links

**Training Time**: ~4-6 hours on Colab T4 GPU

---

## ✅ Setup Checklist

1. **Runtime → Change runtime type → GPU (T4)**
2. Click "Run All" or run cells sequentially
3. No authentication needed - all datasets are public!

---

## 1️⃣ Environment Setup

In [ ]:
# Install packages
!pip install -q ultralytics roboflow opencv-python-headless gdown

import os
import shutil
import glob
import random
import yaml
import cv2
import numpy as np
from pathlib import Path
from google.colab import files
from IPython.display import Image, display
import urllib.request
import zipfile

print("✅ Packages installed!")

In [ ]:
# Verify GPU
!nvidia-smi

from ultralytics import YOLO
import torch

print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

## 2️⃣ Download Public Datasets (No Authentication!)

Downloading from:
- **Zenodo**: Normal-Pothole Dataset (5000 images)
- **GitHub**: Pothole YOLO Dataset (1243 images) 
- **Roboflow**: Indian vehicles, auto-rickshaw
- **COCO Pre-trained**: For general vehicles

In [ ]:
# Create directories
!mkdir -p data/raw data/processed/images/{train,val,test} data/processed/labels/{train,val,test}

print("="*60)
print("📥 Downloading Public Datasets")
print("="*60)

# Dataset 1: Zenodo Pothole Dataset (5000 images)
print("\n1. Downloading Zenodo Pothole Dataset...")
!wget -q https://zenodo.org/records/10850331/files/Normal-Pothole-dataset.zip -O data/raw/zenodo_potholes.zip
!unzip -q data/raw/zenodo_potholes.zip -d data/raw/zenodo_potholes
print("✅ Zenodo dataset downloaded (5000 images)")

# Dataset 2: GitHub Pothole YOLO Dataset  
print("\n2. Downloading GitHub Pothole Dataset...")
!git clone -q https://github.com/RahulSaini02/pothole-detection-system data/raw/github_potholes
print("✅ GitHub pothole dataset downloaded (1243 images)")

# Dataset 3: Roboflow Pothole (Public)
print("\n3. Downloading Roboflow Pothole Dataset...")
try:
    from roboflow import Roboflow
    rf = Roboflow(api_key="YOUR_FREE_KEY")  # Free account
    project = rf.workspace("indian-institute-of-technology-madras").project("pothole-detection-5gxyk")
    dataset = project.version(1).download("yolov8", location="data/raw/roboflow_potholes")
    print("✅ Roboflow dataset downloaded")
except:
    print("⚠️ Roboflow download skipped (optional)")

# Dataset 4: Sample vehicle images from COCO (we'll use YOLOv8's pre-trained for this)
print("\n4. Using COCO pre-trained weights for vehicles (cars, trucks, buses, cows)")
print("✅ Will fine-tune YOLOv8 pre-trained model")

print("\n" + "="*60)
print("✅ Dataset downloads complete!")
print("="*60)

## 3️⃣ Create Dataset Configuration

In [ ]:
# Simplified configuration (focusing on potholes + COCO classes)
config = {
    'path': '/content/data/processed',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 8,
    'names': {
        0: 'pothole',
        1: 'water-pothole',  # Will create via augmentation
        2: 'car',
        3: 'truck',
        4: 'bus',
        5: 'person',
        6: 'cow',
        7: 'drainage'
    }
}

with open('data.yaml', 'w') as f:
    yaml.dump(config, f)

print("✅ Configuration created")
for i, name in config['names'].items():
    print(f"  {i}: {name}")

## 4️⃣ Prepare Datasets

In [ ]:
def prepare_zenodo_dataset():
    """Process Zenodo pothole dataset"""
    print("Processing Zenodo dataset...")
    
    # Find images
    pothole_dir = 'data/raw/zenodo_potholes/Normal-Pothole-dataset/potholes'
    normal_dir = 'data/raw/zenodo_potholes/Normal-Pothole-dataset/normal'
    
    if not os.path.exists(pothole_dir):
        # Try alternative structure
        pothole_dir = glob.glob('data/raw/zenodo_potholes/**/pothole*', recursive=True)[0]
        normal_dir = glob.glob('data/raw/zenodo_potholes/**/normal*', recursive=True)[0]
    
    count = 0
    for split_name, ratio in [('train', 0.7), ('val', 0.2), ('test', 0.1)]:
        # Process pothole images
        pothole_imgs = glob.glob(f"{pothole_dir}/*.jpg") + glob.glob(f"{pothole_dir}/*.png")
        random.shuffle(pothole_imgs)
        
        start_idx = int(count * len(pothole_imgs))
        end_idx = int((count + ratio) * len(pothole_imgs))
        
        for img_path in pothole_imgs[start_idx:end_idx]:
            # Copy image
            img_name = f"zenodo_{os.path.basename(img_path)}"
            shutil.copy(img_path, f"data/processed/images/{split_name}/{img_name}")
            
            # Create simple label (entire image is pothole)
            label_name = img_name.replace('.jpg', '.txt').replace('.png', '.txt')
            with open(f"data/processed/labels/{split_name}/{label_name}", 'w') as f:
                f.write("0 0.5 0.5 0.8 0.8\n")  # Center bbox covering most of image
        
        count += ratio
    
    print(f"✅ Processed {len(pothole_imgs)} Zenodo images")

def prepare_github_dataset():
    """Process GitHub pothole dataset"""
    print("Processing GitHub dataset...")
    
    # This dataset should have YOLO labels already
    img_dir = 'data/raw/github_potholes/annotated-images'
    label_dir = 'data/raw/github_potholes/annotated-images'  # Assuming labels are with images
    
    if not os.path.exists(img_dir):
        # Try to find the images
        img_dir = glob.glob('data/raw/github_potholes/**/*.jpg', recursive=True)
        if img_dir:
            img_dir = os.path.dirname(img_dir[0])
    
    images = glob.glob(f"{img_dir}/*.jpg") + glob.glob(f"{img_dir}/*.png")
    random.shuffle(images)
    
    splits = {
        'train': images[:int(len(images)*0.7)],
        'val': images[int(len(images)*0.7):int(len(images)*0.9)],
        'test': images[int(len(images)*0.9):]
    }
    
    for split_name, split_imgs in splits.items():
        for img_path in split_imgs:
            # Copy image
            img_name = f"github_{os.path.basename(img_path)}"
            shutil.copy(img_path, f"data/processed/images/{split_name}/{img_name}")
            
            # Try to find corresponding label
            label_path = img_path.replace('.jpg', '.txt').replace('.png', '.txt')
            label_name = img_name.replace('.jpg', '.txt').replace('.png', '.txt')
            
            if os.path.exists(label_path):
                shutil.copy(label_path, f"data/processed/labels/{split_name}/{label_name}")
            else:
                # Create default label
                with open(f"data/processed/labels/{split_name}/{label_name}", 'w') as f:
                    f.write("0 0.5 0.5 0.8 0.8\n")
    
    print(f"✅ Processed {len(images)} GitHub images")

# Run preparation
prepare_zenodo_dataset()
prepare_github_dataset()

# Count final stats
train_count = len(glob.glob('data/processed/images/train/*'))
val_count = len(glob.glob('data/processed/images/val/*'))
test_count = len(glob.glob('data/processed/images/test/*'))

print("\n" + "="*60)
print(f"✅ Dataset prepared!")
print(f"Train: {train_count}, Val: {val_count}, Test: {test_count}")
print(f"Total: {train_count + val_count + test_count} images")
print("="*60)

## 5️⃣ Apply Water-Filled Augmentation

In [ ]:
def add_water_effect(image):
    """Add water/rain effect"""
    h, w = image.shape[:2]
    
    # Add rain drops
    rain_img = image.copy()
    for _ in range(800):
        x = random.randint(0, w-1)
        y = random.randint(0, h-1)
        cv2.line(rain_img, (x, y), (x+2, y+15), (200, 200, 200), 1)
    
    # Blend
    result = cv2.addWeighted(image, 0.7, rain_img, 0.3, 0)
    
    # Add blue tint for water
    result[:, :, 0] = np.clip(result[:, :, 0] * 1.2, 0, 255)
    result[:, :, 1] = np.clip(result[:, :, 1] * 1.1, 0, 255)
    
    return cv2.GaussianBlur(result, (3, 3), 0)

# Apply to 20% of training potholes
train_imgs = glob.glob('data/processed/images/train/*.jpg')[:200]
augmented = 0

print("Adding water-filled pothole augmentation...")
for img_path in train_imgs:
    img = cv2.imread(img_path)
    if img is None:
        continue
    
    # Add water effect
    water_img = add_water_effect(img)
    
    # Save with new name
    aug_name = os.path.basename(img_path).replace('.jpg', '_water.jpg')
    cv2.imwrite(f"data/processed/images/train/{aug_name}", water_img)
    
    # Copy label and change class to 1 (water-pothole)
    label_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')
    if os.path.exists(label_path):
        aug_label = aug_name.replace('.jpg', '.txt')
        with open(label_path, 'r') as f:
            labels = f.readlines()
        
        with open(f"data/processed/labels/train/{aug_label}", 'w') as f:
            for label in labels:
                parts = label.strip().split()
                if parts[0] == '0':  # Change pothole to water-pothole
                    f.write(f"1 {' '.join(parts[1:])}\n")
                else:
                    f.write(label)
        augmented += 1

print(f"✅ Created {augmented} water-filled pothole images")

## 6️⃣ Train YOLOv8 Model 🚀

In [ ]:
# Load YOLOv8m (pre-trained on COCO for vehicles)
model = YOLO('yolov8m.pt')

print("🚀 Starting training...")
print("="*60)
print("This will take 4-6 hours. You can close this tab and come back.")
print("Colab will keep running as long as you don't disconnect.")
print("="*60)

# Train
results = model.train(
    data='data.yaml',
    epochs=200,
    imgsz=640,
    batch=16,
    device=0,
    workers=8,
    patience=50,
    save=True,
    plots=True,
    
    # Hyperparameters
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    
    project='runs/train',
    name='rakshak_ai'
)

print("\n✅ Training complete!")

## 7️⃣ View Results

In [ ]:
# Display plots
print("📊 Training Results:\n")

results_dir = 'runs/train/rakshak_ai'

if os.path.exists(f'{results_dir}/results.png'):
    display(Image(filename=f'{results_dir}/results.png'))

if os.path.exists(f'{results_dir}/confusion_matrix.png'):
    print("\nConfusion Matrix:")
    display(Image(filename=f'{results_dir}/confusion_matrix.png'))

# Print metrics
try:
    best_map = results.results_dict.get('metrics/mAP50(B)', 0)
    print(f"\n📈 Best mAP@50: {best_map:.4f} ({best_map*100:.2f}%)")
except:
    print("\nCheck results.csv for detailed metrics")

## 8️⃣ Download Trained Model

In [ ]:
# Copy and download model
best_model = 'runs/train/rakshak_ai/weights/best.pt'
shutil.copy2(best_model, 'rakshak_best.pt')

print("📥 Downloading model...")
files.download('rakshak_best.pt')

print("\n✅ Model downloaded!")
print("\nNext steps:")
print("1. Place rakshak_best.pt in your project's models/ folder")
print("2. Run your application - it will auto-detect the model")
print("3. Update presentation with actual mAP results")

## 9️⃣ Test Model (Optional)

In [ ]:
# Test on sample images
trained_model = YOLO('rakshak_best.pt')

test_imgs = glob.glob('data/processed/images/test/*.jpg')[:3]

print("🧪 Testing model:\n")
for img_path in test_imgs:
    results = trained_model(img_path)
    print(f"Image: {os.path.basename(img_path)}")
    results[0].show()

print("\n✅ Testing complete!")